# Models — does review text help predict rating?

Sibling of `01_models.ipynb`, but the target is **`rating`** instead of
`retail`. Mirrors the same structured baseline + text feature blocks, so any
metric difference is attributable to the added block:

- `features_basic.parquet`          (from `etl/04_feature_engineering.ipynb`)
- `features_keywords.parquet`       (from `etl/05_nlp_keywords.ipynb`)
- `features_embeddings_PCA.parquet` (from `etl/06_nlp_embeddings.ipynb` → `etl/07_PCA_reduction.ipynb`)

joined on `wine_id`:

- **Model 1** — base structured features only (now incl. `retail` as a predictor)
- **Model 2** — base + 8 keyword aroma densities
- **Model 3** — base + 20 embedding principal components

Two differences vs `01`:
- `retail` flips from target to **feature**; `rating` is the new target.
- We still **omit extreme prices** — rows are filtered to the 2nd–90th
  retail percentile (same `$11–$80` band as `01`), here as a data filter
  rather than a target trim. This also drops rows with no `retail`.

In [9]:
import numpy as np
import pandas as pd
import itables
from itables import show
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

itables.options.columnDefs = [{"className": "dt-left", "targets": "_all"}]

FEATURES_BASIC_PATH = r"..\..\.data\features_basic.parquet"
KEYWORDS_PATH = r"..\..\.data\features_keywords.parquet"
EMBEDDINGS_PCA_PATH = r"..\..\.data\features_embeddings_PCA.parquet"

## Load features

In [10]:
features = pd.read_parquet(FEATURES_BASIC_PATH)
keywords = pd.read_parquet(KEYWORDS_PATH)
embeddings_pca = pd.read_parquet(EMBEDDINGS_PCA_PATH)
print(f"features: {features.shape}  |  keywords: {keywords.shape}  |  embeddings_pca: {embeddings_pca.shape}")

target = "rating"

# retail is now a predictor (was the target in 01); rating drops out of the features.
base_features = [
    "retail", "alcohol", "bottle_size", "vintage", "case_production",
    "country_ord", "wine_type_ord", "state_ord", "company_ord",
    "appellation_ord", "varietal_label_ord", "age_at_review",
]
base_features_excl_retail = [f for f in base_features if f != "retail"]

kw_features  = [c for c in keywords.columns if c.startswith("kw_") and not c.endswith("_count")]
pca_features = [c for c in embeddings_pca.columns if c.startswith("pca_")]

df = (
    features
    .merge(keywords[["wine_id"] + kw_features], on="wine_id", how="left")
    .merge(embeddings_pca[["wine_id"] + pca_features], on="wine_id", how="left")
)
assert len(df) == len(features), "merge changed row count — wine_id not unique?"
print(f"merged: {df.shape}")
print(f"basic features ({len(base_features)}): {base_features}")
print(f"aroma features ({len(kw_features)}): {kw_features}")
print(f"emb-PCA features ({len(pca_features)}): {pca_features}")

features: (135192, 15)  |  keywords: (135192, 17)  |  embeddings_pca: (135192, 21)
merged: (135192, 43)
basic features (12): ['retail', 'alcohol', 'bottle_size', 'vintage', 'case_production', 'country_ord', 'wine_type_ord', 'state_ord', 'company_ord', 'appellation_ord', 'varietal_label_ord', 'age_at_review']
aroma features (8): ['kw_fruity', 'kw_tannic', 'kw_acidic', 'kw_oaky', 'kw_sweet', 'kw_body', 'kw_earthy', 'kw_floral']
emb-PCA features (20): ['pca_00', 'pca_01', 'pca_02', 'pca_03', 'pca_04', 'pca_05', 'pca_06', 'pca_07', 'pca_08', 'pca_09', 'pca_10', 'pca_11', 'pca_12', 'pca_13', 'pca_14', 'pca_15', 'pca_16', 'pca_17', 'pca_18', 'pca_19']


## Train & compare

All models train on the *same* rows and split (`random_state=42`); they
differ only in the text block added on top of `base`.

Row filter: drop rows with no `rating` (none, in practice), then **omit
extreme prices** by keeping only `retail` in the 2nd–90th percentile — the
same band as `01`, applied here as a data filter.

In [11]:
# one big model_df with all feature blocks; same rows for every model
model_df = df[base_features + kw_features + pca_features + [target]].dropna(subset=[target])

# omit extreme prices: keep rows within the 2nd-90th retail percentile
low, high = model_df["retail"].quantile([0.02, 0.90])
model_df = model_df[model_df["retail"].between(low, high)]
print(f"Kept retail ${low:.2f} - ${high:.2f}  ({len(model_df):,} rows)  | target = {target}")

# function to train and evaluate a model given a list of features
def train_eval(feature_list):
    X, y = model_df[feature_list], model_df[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = xgb.XGBRegressor(random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    return model, {
        "n_features": len(feature_list),
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "MAE":  mean_absolute_error(y_test, pred),
        "R2":   r2_score(y_test, pred),
    }

Kept retail $11.00 - $80.00  (113,220 rows)  | target = rating


### With `retail` (full base)

In [12]:
# train and evaluate models with retail prices
model1, m1 = train_eval(base_features)
model2, m2 = train_eval(base_features + kw_features)
model3, m3 = train_eval(base_features + pca_features)

# compare models
results = pd.DataFrame([
    {"model": "1: baseline",            **m1},
    {"model": "2: baseline + kw",       **m2},
    {"model": "3: baseline + emb-PCA",  **m3},
])
results["dR2_vs_base"] = results["R2"] - m1["R2"]
results.round(4)

,model,n_features,RMSE,MAE,R2,dR2_vs_base
0,1: baseline,12,1.8317,1.4381,0.4517,0.0000
1,2: baseline + kw,20,1.7254,1.3492,0.5135,0.0618
2,3: baseline + emb-PCA,32,1.7875,1.3992,0.4778,0.0261


### Without `retail`

Drops `retail` from `base` and retrains all three models (`retail` still drives
the extreme-price row filter, just isn't a predictor). Shows how much rating
prediction depends on price, and whether the text blocks (`kw` / `emb-PCA`)
compensate. `dR2_vs_base` here is relative to the no-retail baseline (model 4).

In [13]:
# Same three models, but with `retail` dropped from the feature set.
# (retail is still used to omit extreme prices in the row filter — just not as a
# predictor.) retail is the strongest structured predictor of rating, so this
# shows how much the score model leans on price, and whether the text blocks
# recover signal in its absence.
model4, m4 = train_eval(base_features_excl_retail)
model5, m5 = train_eval(base_features_excl_retail + kw_features)
model6, m6 = train_eval(base_features_excl_retail + pca_features)

results_excl = pd.DataFrame([
    {"model": "4: no-retail base",       **m4},
    {"model": "5: no-retail + kw",       **m5},
    {"model": "6: no-retail + emb-PCA",  **m6},
])
results_excl["dR2_vs_base"] = results_excl["R2"] - m4["R2"]
results_excl.round(4)

,model,n_features,RMSE,MAE,R2,dR2_vs_base
0,4: no-retail base,11,1.9725,1.5586,0.3641,0.0000
1,5: no-retail + kw,19,1.8550,1.4586,0.4376,0.0736
2,6: no-retail + emb-PCA,31,1.9357,1.5250,0.3876,0.0235


## Feature importance — model 3

Where the 20 embedding PCs land relative to the structured features (incl.
`retail`), and how much the `emb-PCA` block contributes vs `base`.

In [14]:
imp = (
    pd.DataFrame({"feature": base_features + pca_features, "importance": model3.feature_importances_})
    .assign(group=lambda d: np.where(d["feature"].str.startswith("pca_"), "emb-PCA", "base"))
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
    .assign(importance=lambda d: d["importance"].round(3))
)

print("total importance by block:")
print(imp.groupby("group")["importance"].sum().round(3).to_string())
show(imp)

total importance by block:
group
base       0.651
emb-PCA    0.350


Loading ITables v2.7.3 from the internet... (need help?)


## Conclusion

Read the table above (`dR2_vs_base`): a positive value means the text block
adds rating signal beyond the structured features. Expectations to sanity-check:

- `rating` (80–100) is a narrower, noisier target than price, so absolute R²
  will differ from `01` — compare *relative* gains, not levels.
- Review prose is written by the same reviewer who assigns the score, so the
  embeddings may carry **more** signal for `rating` than they did for `retail`.
  If model 3 lifts `rating` but not `retail`, that's an interesting asymmetry
  worth noting in the report.

Same follow-ups as `01`: tune `N_COMPONENTS` (07), regularise, MLflow CV (Sprint 7).